## Principal Component Analysis (PCA)

**Principal Component Analysis (PCA)** is a mathematical technique that simplifies complex, high-dimensional datasets by finding new coordinate axes — called principal components (PCs) — that capture the maximum variance in the data.

---

### Step-by-step intuition

1. **Start with standardized features**  
   PCA begins by centering and scaling the data so that all variables have mean = 0 and variance = 1.  
   (Without this, variables measured in different units — e.g., mRNA counts vs. metabolite concentrations — would dominate unfairly.)

2. **Find the directions of greatest variance**  
   PCA looks for a new axis (a linear combination of the original features) that captures as much variation in the data as possible. "Weights" here are analagous to the "slopes/betas" in regression.

   Mathematically, it solves for weight vectors \( $\mathbf{w}_1$, $\mathbf{w}_2$, … \) that maximize:

   $$
   \text{Var}(X\mathbf{w}) \quad \text{subject to } \|\mathbf{w}\| = 1
   $$

3. **Orthogonal components**  
   After finding the first principal component (PC1), PCA finds another axis (PC2) that:
   - Captures the next-largest amount of variance  
   - Is **orthogonal** (uncorrelated) to PC1

4. **Project the data**  
   Each sample’s coordinates along PC1, PC2, … are computed as:

   $$
   Z = X \mathbf{W}
   $$

   where  
   • \($ X $\) = standardized data matrix  
   • \( $\mathbf{W} $\) = matrix of eigenvectors (principal axes)  
   • \($ Z $\) = transformed data in the new PCA space

---

### Connection to linear algebra

- PCA is computed using **eigen-decomposition** or **Singular Value Decomposition (SVD)**.  
- The eigenvectors of the covariance matrix \($ X^\top X $\) define the directions of principal components.  
- The eigenvalues tell how much variance each component explains.

---

### What PCA gives you

| Output | Meaning |
|---------|----------|
| **PC scores** | The coordinates of each sample in the new PCA space (used for plotting) |
| **Loadings** | How strongly each original feature contributes to each PC |
| **Explained variance ratio** | Fraction of total variance captured by each component |

---

### What's it doing?

> PCA rotates the data to a new set of axes that best describe how the samples vary. The first few principal components usually capture most of the biological or experimental structure, letting us visualize patterns, detect outliers, and reduce noise.


In [ ]:
#@title Setup & synthetic 2D data
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

# latent factor -> creates correlation between x and y
n = 200
z = rng.normal(0, 1, n)
eps_x = rng.normal(0, 0.6, n)
eps_y = rng.normal(0, 0.8, n)

# Make x,y with a linear relationship but different noise scales
x = 2.0*z + eps_x
y = 1.2*x + 3.0 + eps_y  # true slope ~1.2, intercept ~3.0

X = np.column_stack([x, y])
x_mean, y_mean = x.mean(), y.mean()
Xc = X - X.mean(axis=0)  # centered for PCA

In [ ]:
#@title Fit regression & PCA; compute lines
from sklearn.linear_model import LinearRegression
from numpy.linalg import svd

# Linear regression (y ~ x)
lm = LinearRegression()
lm.fit(x.reshape(-1,1), y)
b_lr = lm.intercept_
w_lr = lm.coef_[0]

# PCA via SVD on centered data (Xc)
# Xc = U S V^T, rows are samples, columns are [x, y]
U, S, Vt = svd(Xc, full_matrices=False)
pc1 = Vt[0]             # first principal component direction in (x,y) coords
vx, vy = pc1            # direction vector
# slope of PC1 line in x-y plane (avoid divide by zero)
slope_pc = vy / vx if np.abs(vx) > 1e-12 else np.inf
# PC1 line passes through the mean (x_mean, y_mean)
b_pc = y_mean - slope_pc * x_mean if np.isfinite(slope_pc) else None

# Prepare a plotting range
xs = np.linspace(x.min()-0.5, x.max()+0.5, 200)
y_lr = b_lr + w_lr*xs
y_pc = (b_pc + slope_pc*xs) if np.isfinite(slope_pc) else np.full_like(xs, np.nan)

print(f"Regression:   y = {w_lr:.3f} * x + {b_lr:.3f}")
if np.isfinite(slope_pc):
    print(f"PCA (PC1):    y = {slope_pc:.3f} * x + {b_pc:.3f}  (line through data mean)")
else:
    print("PCA (PC1):    vertical line through x = x_mean")


In [ ]:
#@title Plot data with regression line & PC1 line, plus residual segments
plt.figure(figsize=(7,6))
plt.scatter(x, y, s=18, alpha=0.7, label="data")

# Regression line
plt.plot(xs, y_lr, lw=2, label="Linear regression (minimize vertical error)")

# PC1 line (variance-maximizing direction)
if np.isfinite(slope_pc):
    plt.plot(xs, y_pc, lw=2, linestyle="--", label="PCA PC1 (variance direction)")
else:
    plt.axvline(x_mean, lw=2, linestyle="--", label="PCA PC1 (vertical line)")

# Show residual segments for a small random subset to avoid clutter
idx = rng.choice(len(x), size=15, replace=False)

# Vertical residuals to regression line (blue, dotted)
for i in idx:
    y_hat = b_lr + w_lr*x[i]
    plt.plot([x[i], x[i]], [y[i], y_hat], color="#2563eb", ls=":", lw=1.8)

# Orthogonal residuals to PC1 line (red, dotted)
if np.isfinite(slope_pc):
    # Unit direction vector along PC1 and its normal
    v = np.array([vx, vy])
    v = v / (np.linalg.norm(v) + 1e-12)
    nvec = np.array([-v[1], v[0]])  # normal to PC1
    mu = np.array([x_mean, y_mean])

    for i in idx:
        p = np.array([x[i], y[i]])
        # projection point on PC1:
        t = np.dot(p - mu, v)
        proj = mu + t * v
        plt.plot([p[0], proj[0]], [p[1], proj[1]], color="#dc2626", ls=":", lw=1.8)

plt.xlabel("x"); plt.ylabel("y")
plt.title("Regression vs PCA: two different projections")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#@title Compare "errors": vertical vs orthogonal-to-PC1
# Vertical residuals to regression line
y_hat_lr = b_lr + w_lr*x
mse_vertical = np.mean((y - y_hat_lr)**2)

# Orthogonal distances to PC1 line
if np.isfinite(slope_pc):
    v = np.array([vx, vy])
    v = v / (np.linalg.norm(v) + 1e-12)
    mu = np.array([x_mean, y_mean])
    P = np.column_stack([x, y])
    # vector from mean to each point
    P0 = P - mu
    # parallel component length along v
    t = P0 @ v
    proj = mu + np.outer(t, v)
    orth_dists = np.linalg.norm(P - proj, axis=1)
    mean_orth = orth_dists.mean()
else:
    # vertical PC1: orthogonal distance is horizontal distance to x_mean
    mean_orth = np.mean(np.abs(x - x_mean))

print(f"MSE (vertical, regression): {mse_vertical:.3f}")
print(f"Mean orthogonal distance to PC1: {mean_orth:.3f}")


## **Why not use PC1 for the model (Y ~ PC1*x + error)?**

In [ ]:
#@title Optional: Principal Component Regression (using PC1 as 1-D summary of X)
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Build a 2D feature matrix for PCR (Xc already centered)
# PC1 scores: U[:,0]*S[0] equals Xc @ pc1   (up to sign)
pc1_scores = Xc @ pc1  # 1-D representation of (x,y)

# Compare y ~ x   vs   y ~ PC1
X_train, X_test, y_train, y_test = train_test_split(
    x.reshape(-1,1), y, test_size=0.3, random_state=42
)
lm_x = LinearRegression().fit(X_train, y_train)
pred_x = lm_x.predict(X_test)

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    pc1_scores.reshape(-1,1), y, test_size=0.3, random_state=42
)
lm_p = LinearRegression().fit(X_train_p, y_train_p)
pred_p = lm_p.predict(X_test_p)

print("y ~ x     -> R^2 = %.3f, RMSE = %.3f" % (r2_score(y_test, pred_x), np.sqrt(mean_squared_error(y_test, pred_x))))
print("y ~ PC1   -> R^2 = %.3f, RMSE = %.3f" % (r2_score(y_test_p, pred_p), np.sqrt(mean_squared_error(y_test_p, pred_p))))


In [ ]:
#@title PCA rotation + regressions + PCs vs y (2×2 panel)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# --- Simulate correlated data ---
rng = np.random.default_rng(42)
n = 200
x1 = rng.normal(0, 5, size=n)                 # high variance, unrelated to y
x2 = 0.5 * x1 + rng.normal(0, 1, size=n)      # correlated with x1
X  = np.column_stack([x1, x2])
y = 3*x2 + rng.normal(0, 1, size=n)           # response depends on x2

# --- PCA ---
pca = PCA(n_components=2)
PC = pca.fit_transform(X)
mu = X.mean(axis=0)
scale = 2.5
pc1 = scale * np.sqrt(pca.explained_variance_ratio_[0]) * pca.components_[0]
pc2 = scale * np.sqrt(pca.explained_variance_ratio_[1]) * pca.components_[1]

# --- Regressions ---
lm1 = LinearRegression().fit(x1.reshape(-1,1), y)
lm2 = LinearRegression().fit(x2.reshape(-1,1), y)
r2_1 = r2_score(y, lm1.predict(x1.reshape(-1,1)))
r2_2 = r2_score(y, lm2.predict(x2.reshape(-1,1)))

# Regress y on PCs
lm_pc1 = LinearRegression().fit(PC[:, [0]], y)
lm_pc2 = LinearRegression().fit(PC[:, [1]], y)
r2_pc1 = r2_score(y, lm_pc1.predict(PC[:, [0]]))
r2_pc2 = r2_score(y, lm_pc2.predict(PC[:, [1]]))

print(f"R²(y~x₁)={r2_1:.3f}, R²(y~x₂)={r2_2:.3f}")
print(f"R²(y~PC1)={r2_pc1:.3f}, R²(y~PC2)={r2_pc2:.3f}")
print(f"Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.2f}, PC2={pca.explained_variance_ratio_[1]:.2f}")

# --- 2×2 panel ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Top-left: Original data + PCA arrows
ax = axes[0, 0]
ax.scatter(x1, x2, c=y, cmap="viridis", s=25, alpha=0.8)
ax.arrow(mu[0], mu[1], pc1[0], pc1[1], width=0.03, color="crimson", alpha=0.9, length_includes_head=True)
ax.arrow(mu[0], mu[1], pc2[0], pc2[1], width=0.03, color="royalblue", alpha=0.9, length_includes_head=True)
ax.text(mu[0]+pc1[0]*1.1, mu[1]+pc1[1]*1.1, "PC1", color="crimson")
ax.text(mu[0]+pc2[0]*1.1, mu[1]+pc2[1]*1.1, "PC2", color="royalblue")
ax.set_aspect('equal')
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")
ax.set_title("Correlated features with PCA rotation")
ax.grid(alpha=0.3)

# Top-right: PCs vs y
ax = axes[0, 1]
ax.scatter(PC[:, 0], y, c="crimson", s=25, alpha=0.7, label=f"PC1 (R²={r2_pc1:.2f})")
ax.scatter(PC[:, 1], y, c="royalblue", s=25, alpha=0.7, label=f"PC2 (R²={r2_pc2:.2f})")
ax.set_xlabel("Principal Component value")
ax.set_ylabel("y (response)")
ax.legend()
ax.set_title("y vs PC1 and PC2")
ax.grid(alpha=0.3)

# Bottom-left: y ~ x₁
ax = axes[1, 0]
x_line = np.linspace(x1.min(), x1.max(), 100).reshape(-1, 1)
ax.scatter(x1, y, c="teal", s=25, alpha=0.6)
ax.plot(x_line, lm1.predict(x_line), color="crimson", lw=2, label=f"OLS (R²={r2_1:.2f})")
ax.set_xlabel("x₁")
ax.set_ylabel("y")
ax.set_title("Regression: y ~ x₁")
ax.legend()
ax.grid(alpha=0.3)

# Bottom-right: y ~ x₂
ax = axes[1, 1]
x_line = np.linspace(x2.min(), x2.max(), 100).reshape(-1, 1)
ax.scatter(x2, y, c="teal", s=25, alpha=0.6)
ax.plot(x_line, lm2.predict(x_line), color="crimson", lw=2, label=f"OLS (R²={r2_2:.2f})")
ax.set_xlabel("x₂")
ax.set_ylabel("y")
ax.set_title("Regression: y ~ x₂")
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle("Why PC1 (high variance) may not be the best predictor of y", fontsize=15)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# --- Loadings table ---
feature_names = ["x₁","x₂"]
loadings_df = pd.DataFrame(pca.components_.T, index=feature_names, columns=["PC1","PC2"])
explained_df = pd.DataFrame({
    "Eigenvalue": pca.explained_variance_,
    "ExplainedVarianceRatio": pca.explained_variance_ratio_
}, index=["PC1","PC2"])

print("\n=== PCA Loadings (eigenvectors) ===")
display(loadings_df.round(3))
print("\n=== Explained Variance ===")
display(explained_df.round(3))


#Extending to multiple features/predictors/variables
##Do multivariate regression on full feature set or do dimensionality reduction and regression on PCs?
###Multiple PCs example 5 varables to predict enzyme activity

* GeneA_expression     
* GeneB_expression     
* Metabolite1_concentration    
* StressMarker_level
* SignalProtein_level




In [ ]:
#@title Simulated biological dataset (5 predictors + target)
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(123)
n = 400

# latent biological factors
metabolic_state = rng.normal(0, 1.0, n)
stress_response = rng.normal(0, 1.0, n)
signaling_state = rng.normal(0, 1.0, n)

# generate correlated biological features
GeneA_expression     = 2.0*metabolic_state + 0.5*stress_response + rng.normal(0, 0.5, n)
GeneB_expression     = 1.5*metabolic_state - 0.3*stress_response + rng.normal(0, 0.5, n)
Metabolite1_conc     = 0.2*metabolic_state + 1.8*stress_response + rng.normal(0, 0.6, n)
StressMarker_level   = -0.5*metabolic_state + 0.4*signaling_state + rng.normal(0, 0.7, n)
SignalProtein_level  = 0.1*stress_response + 1.0*signaling_state + rng.normal(0, 0.7, n)

# stack into feature matrix
X = np.column_stack([GeneA_expression, GeneB_expression,
                     Metabolite1_conc, StressMarker_level,
                     SignalProtein_level])

# true biological relationship -> enzyme activity
true_w = np.array([1.2, 0.0, 0.8, 0.4, 0.0])
enzyme_activity = X @ true_w + 3.0 + rng.normal(0, 0.8, n)

# name columns
cols = ["GeneA_expression","GeneB_expression",
        "Metabolite1_conc","StressMarker_level",
        "SignalProtein_level","Enzyme_activity"]

# build DataFrame and save
df = pd.DataFrame(np.column_stack([X, enzyme_activity]), columns=cols)
df.to_csv("simulated_biology_dataset.csv", index=False)
print(f"✅ Saved simulated_biology_dataset.csv with shape {df.shape}")
df.head()


In [ ]:
#@title Full 5-feature regression summary
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# --- Load the dataset if not already in memory ---
try:
    df
    assert all(col in df.columns for col in [
        "GeneA_expression","GeneB_expression","Metabolite1_conc",
        "StressMarker_level","SignalProtein_level","Enzyme_activity"
    ])
except Exception:
    raise ValueError("Dataset 'df' not found. Please run the cell that creates 'df' first.")

# --- Separate predictors and target ---
X = df[["GeneA_expression","GeneB_expression",
        "Metabolite1_conc","StressMarker_level","SignalProtein_level"]].values
y = df["Enzyme_activity"].values

# --- Fit full regression model ---
lm = LinearRegression().fit(X, y)
y_pred = lm.predict(X)

# --- Compute metrics ---
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

# --- Print summary ---
print("=== Multiple Linear Regression: Enzyme_activity ~ all 5 predictors ===\n")
#print("\nCoefficients:")
#for name, coef in zip(df.columns[:-1], lm.coef_):
#    print(f"  {name:20s} {coef:10.4f}")

print("\nModel fit statistics:")
print(f"  R²     = {r2:.4f}")
print(f"  MSE    = {mse:.4f}")
print(f"  RMSE   = {rmse:.4f}")
print(f"  n      = {len(y)},  p = {X.shape[1]}")

print(f"\nIntercept: {lm.intercept_:.3f}")

# --- Optional: tidy summary table ---
summary_df = pd.DataFrame({
    "Predictor": df.columns[:-1],
    "Coefficient": lm.coef_,
})
display(summary_df.round(4))


In [ ]:
# @title 3D regressions: 2-predictor pairs, ranked by R²
import numpy as np, math
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from itertools import combinations
from sklearn.linear_model import LinearRegression

# ---- config ----
response_col = "Enzyme_activity"
top_k = 6    # number of top pairs to plot (max = 10 for 5 predictors)

# ---- extract features / response ----
feature_cols = [c for c in df.columns if c != response_col]
X_all = df[feature_cols]
y = df[response_col].values

# ---- fit all 2D models and rank by R² ----
pairs = list(combinations(feature_cols, 2))
ranked = []
models = {}
for f1, f2 in pairs:
    Xpair = df[[f1, f2]].values
    lm = LinearRegression().fit(Xpair, y)
    r2 = lm.score(Xpair, y)
    ranked.append((r2, f1, f2))
    models[(f1, f2)] = lm

ranked.sort(reverse=True)
nplots = min(top_k, len(ranked))
ncols = 3
nrows = math.ceil(nplots / ncols)

# ---- plot grid of 3D scatter + fitted plane ----
fig = plt.figure(figsize=(5.2 * ncols, 4.4 * nrows))

for idx, (r2, f1, f2) in enumerate(ranked[:nplots], start=1):
    ax = fig.add_subplot(nrows, ncols, idx, projection="3d")

    x1 = df[f1].values
    x2 = df[f2].values
    lm = models[(f1, f2)]

    # Scatter: Y on vertical axis
    ax.scatter(x1, x2, y, s=10, c="tab:blue", alpha=0.75)

    # Regression plane z = b0 + b1*x1 + b2*x2
    gx = np.linspace(x1.min(), x1.max(), 25)
    gy = np.linspace(x2.min(), x2.max(), 25)
    GX, GY = np.meshgrid(gx, gy)
    GZ = lm.intercept_ + lm.coef_[0] * GX + lm.coef_[1] * GY
    ax.plot_surface(GX, GY, GZ, alpha=0.4, color="crimson", linewidth=0)

    ax.set_xlabel(f1)
    ax.set_ylabel(f2)
    ax.set_zlabel(response_col)
    ax.set_title(
        f"{response_col} ~ {f1} + {f2}\n"
        f"$R^2$={r2:.3f}, b=({lm.coef_[0]:.2f}, {lm.coef_[1]:.2f})",
        pad=10
    )
    ax.view_init(elev=22, azim=35)

plt.tight_layout()
plt.show()


In [ ]:
#@title **Compute PCA**
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Separate features (X) and target (y)
feature_cols = ["GeneA_expression","GeneB_expression",
                "Metabolite1_conc","StressMarker_level",
                "SignalProtein_level"]
target_col = "Enzyme_activity"

X = df[feature_cols].values
y = df[target_col].values

# Standardize features
Xz = StandardScaler().fit_transform(X)

# Perform PCA
pca = PCA(n_components=2, random_state=42)
PC = pca.fit_transform(Xz)

# Extract PC1 and PC2
pc1, pc2 = PC[:, 0], PC[:, 1]

print("Explained variance by PC1 & PC2:",
      np.round(pca.explained_variance_ratio_, 3))


In [ ]:
#@title **2D PC1 vs PC2 scatter (colored by y)**
plt.figure(figsize=(6.5, 5.5))
sc = plt.scatter(pc1, pc2, c=y, cmap="viridis", s=16, alpha=0.9)
plt.colorbar(sc, label="Enzyme Activity")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PC1 vs PC2 (color = Enzyme Activity)")
plt.tight_layout()
plt.show()


In [ ]:
#@title **Rotatable 3D plot: PC1 vs PC2 vs y (aspect = cube)**
# If running in Colab, ensure plotly is available
try:
    import plotly
except ImportError:
    !pip -q install plotly

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=pc1, y=pc2, z=y,
    mode='markers',
    marker=dict(size=3, opacity=0.9),
    #marker=dict(size=3, color=y, colorscale='Viridis', showscale=True, opacity=0.9),

    name='samples'
))

fig.update_layout(
    title='Rotatable PCA space: PC1 vs PC2 vs Enzyme Activity',
    scene=dict(
        xaxis_title='PC1',
        yaxis_title='PC2',
        zaxis_title='Enzyme Activity',
        aspectmode='cube'   # 🔹 makes the box a cube
    ),
    width=750, height=750
)

fig.show()


In [ ]:
#@title **Compare y ~ (PC1, PC2) vs y ~ X (original 5 features)**
from sklearn.metrics import r2_score, mean_squared_error

# PCR with first 2 PCs
X_pcr = np.column_stack([pc1, pc2])
lm_pcr = LinearRegression().fit(X_pcr, y)
yhat_pcr = lm_pcr.predict(X_pcr)

# Full model with original features
lm_full = LinearRegression().fit(X, y)
yhat_full = lm_full.predict(X)

def report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:16s}  RMSE = {rmse:.3f}   R² = {r2:.3f}")

report("PCR (PC1+PC2)", y, yhat_pcr)
report("Full (5 feats)", y, yhat_full)
print("True weights:", np.round(true_w, 3))
print("Full-model est. weights:", np.round(lm_full.coef_, 3), "  intercept:", round(lm_full.intercept_, 3))


In [ ]:
#@title **PCA fit + loadings table (biological features)**
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Separate features (X) and target (y)
feature_cols = ["GeneA_expression","GeneB_expression",
                "Metabolite1_conc","StressMarker_level",
                "SignalProtein_level"]
target_col = "Enzyme_activity"

X = df[feature_cols].values
y = df[target_col].values

# Standardize features before PCA
scaler = StandardScaler()
Xz = scaler.fit_transform(X)

# PCA -> 2 components
pca = PCA(n_components=2, random_state=42)
scores = pca.fit_transform(Xz)     # sample projections (PC scores)
loadings = pca.components_.T       # shape: (n_features, n_components)

explained = pca.explained_variance_ratio_
print("Explained variance ratio:", np.round(explained, 3))

# Loadings table
loadings_df = pd.DataFrame(loadings, index=feature_cols, columns=["PC1_loading","PC2_loading"])
loadings_df["loading_norm"] = np.sqrt(loadings_df["PC1_loading"]**2 + loadings_df["PC2_loading"]**2)
loadings_df.sort_values("loading_norm", ascending=False)


In [ ]:
#@title **PCA biplot (PC1 vs PC2) with variable arrows**
import matplotlib.pyplot as plt

pc1, pc2 = scores[:,0], scores[:,1]

plt.figure(figsize=(7,6))

# (Optional) color samples by target to connect unsupervised PCs to y
sc = plt.scatter(pc1, pc2, c=y, s=20, cmap="viridis", alpha=0.85)
cbar = plt.colorbar(sc)
cbar.set_label(target_col)

# Arrow scaling so vectors are visible relative to the score spread
# Heuristic: scale arrows to ~40% of the data range
scale = 0.4 * max(pc1.max()-pc1.min(), pc2.max()-pc2.min())

# Draw loading vectors (arrows) from origin
for i, name in enumerate(feature_cols):
    vx = loadings[i,0] * scale
    vy = loadings[i,1] * scale
    plt.arrow(0, 0, vx, vy, color="crimson", width=0.0, head_width=0.07*scale,
              length_includes_head=True, alpha=0.9)
    # Label slightly beyond the arrow tip
    offset = 0.06*scale
    plt.text(vx + offset, vy + offset, name, color="crimson", fontsize=11)

plt.axhline(0, color="#999", lw=1)
plt.axvline(0, color="#999", lw=1)
plt.xlabel(f"PC1 ({explained[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({explained[1]*100:.1f}% var)")
plt.title("PCA Biplot: samples in PC space with variable loadings")
plt.tight_layout()
plt.show()


In [ ]:
#@title **Variable contributions to PC1 and PC2 (absolute loadings)**
abs_load = loadings_df[["PC1_loading","PC2_loading"]].abs().copy()
abs_load.sort_values("PC1_loading", ascending=False, inplace=True)

plt.figure(figsize=(7,4))
plt.bar(abs_load.index, abs_load["PC1_loading"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("|loading|")
plt.title("Absolute loadings on PC1 (variable contribution)")
plt.tight_layout()
plt.show()

abs_load2 = loadings_df[["PC1_loading","PC2_loading"]].abs().copy()
abs_load2.sort_values("PC2_loading", ascending=False, inplace=True)

plt.figure(figsize=(7,4))
plt.bar(abs_load2.index, abs_load2["PC2_loading"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("|loading|")
plt.title("Absolute loadings on PC2 (variable contribution)")
plt.tight_layout()
plt.show()


# **Is 2 PCAs the right number for this dataset?**

In [ ]:
#@title **Extend PCA to 3 components**
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = ["GeneA_expression","GeneB_expression",
                "Metabolite1_conc","StressMarker_level",
                "SignalProtein_level"]
target_col = "Enzyme_activity"

X = df[feature_cols].values
y = df[target_col].values

scaler = StandardScaler()
Xz = scaler.fit_transform(X)

pca3 = PCA(n_components=3, random_state=42)
scores3 = pca3.fit_transform(Xz)
pc1, pc2, pc3 = scores3[:,0], scores3[:,1], scores3[:,2]

print("Explained variance (PC1..PC3):", np.round(pca3.explained_variance_ratio_, 3))


In [ ]:
#@title 3D PCA scatter (PC1, PC2, PC3) colored by Enzyme_activity
try:
    import plotly
except ImportError:
    !pip -q install plotly

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=pc1, y=pc2, z=pc3,
    mode='markers',
    marker=dict(size=3, color=y, colorscale='Viridis', showscale=True, opacity=0.9,
                colorbar=dict(title=target_col)),
    name='samples'
))

fig.update_layout(
    title='4D Scatterplot: PC1 vs PC2 vs PC3 + Enzyme_activity(color)',
    scene=dict(
        xaxis_title=f"PC1 ({pca3.explained_variance_ratio_[0]*100:.1f}% var)",
        yaxis_title=f"PC2 ({pca3.explained_variance_ratio_[1]*100:.1f}% var)",
        zaxis_title=f"PC3 ({pca3.explained_variance_ratio_[2]*100:.1f}% var)",
        aspectmode='cube'   # keep it cubic
    ),
    width=750, height=750
)

fig.show()


In [ ]:
#@title **3D loading arrows (top contributors)**
loadings3 = pca3.components_.T   # shape (n_features, 3)

# scale arrows to ~40% of the score spread
rng_span = max(pc1.max()-pc1.min(), pc2.max()-pc2.min(), pc3.max()-pc3.min())
scale = 0.4 * rng_span

# pick top variables by vector length in PC1..PC3
strength = np.linalg.norm(loadings3, axis=1)
order = np.argsort(strength)[::-1]  # descending
top = order[:5]  # show all five here since you only have 5 vars

for i in top:
    vx, vy, vz = loadings3[i] * scale
    fig.add_trace(go.Scatter3d(
        x=[0, vx], y=[0, vy], z=[0, vz],
        mode='lines+text',
        line=dict(color='crimson', width=6),
        text=[None, feature_cols[i]],
        textposition='top center',
        name=feature_cols[i],
        showlegend=False
    ))

fig.update_layout(title=fig.layout.title.text + " + Loadings")
fig.show()


#**How to decide whether to use dimensionality reduction?**

## - Should we just go with the 5 feature OLS regression, or do regression on PCs?

In [ ]:
#@title Scree + Cumulative Variance for 1-5 PCs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ---- Acquire data (reuse your df if available; otherwise simulate like in your cell) ----
try:
    df
    assert all(col in df.columns for col in [
        "GeneA_expression","GeneB_expression","Metabolite1_conc",
        "StressMarker_level","SignalProtein_level","Enzyme_activity"
    ])
except Exception:
    rng = np.random.default_rng(123)
    n = 400
    metabolic_state = rng.normal(0, 1.0, n)
    stress_response = rng.normal(0, 1.0, n)
    signaling_state = rng.normal(0, 1.0, n)

    GeneA_expression     = 2.0*metabolic_state + 0.5*stress_response + rng.normal(0, 0.5, n)
    GeneB_expression     = 1.5*metabolic_state - 0.3*stress_response + rng.normal(0, 0.5, n)
    Metabolite1_conc     = 0.2*metabolic_state + 1.8*stress_response + rng.normal(0, 0.6, n)
    StressMarker_level   = -0.5*metabolic_state + 0.4*signaling_state + rng.normal(0, 0.7, n)
    SignalProtein_level  = 0.1*stress_response + 1.0*signaling_state + rng.normal(0, 0.7, n)

    X = np.column_stack([GeneA_expression, GeneB_expression,
                         Metabolite1_conc, StressMarker_level,
                         SignalProtein_level])
    true_w = np.array([1.2, 0.0, 0.8, 0.4, 0.0])
    enzyme_activity = X @ true_w + 3.0 + rng.normal(0, 0.8, n)
    cols = ["GeneA_expression","GeneB_expression","Metabolite1_conc",
            "StressMarker_level","SignalProtein_level","Enzyme_activity"]
    df = pd.DataFrame(np.column_stack([X, enzyme_activity]), columns=cols)

feature_cols = ["GeneA_expression","GeneB_expression",
                "Metabolite1_conc","StressMarker_level","SignalProtein_level"]
X = df[feature_cols].values

# ---- Standardize features (recommended before PCA) ----
Xz = StandardScaler().fit_transform(X)

# ---- PCA ----
pca = PCA(n_components=Xz.shape[1]).fit(Xz)
evr = pca.explained_variance_ratio_              # per-component variance explained
cev = np.cumsum(evr)                             # cumulative

# ---- Nicely formatted numbers for printing ----
pct = (evr * 100).round(1)
cpct = (cev * 100).round(1)

print("Explained variance by component (%):", pct.tolist())
print("Cumulative explained variance (%):", cpct.tolist())

# ---- Plot: Scree (left) + Cumulative (right) ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Scree plot
ax = axes[0]
bars = ax.bar(np.arange(1, len(evr)+1), evr, width=0.7, color="#4C78A8", alpha=0.9)
ax.set_xlabel("Principal Component")
ax.set_ylabel("Proportion of variance explained")
ax.set_title("Scree Plot")
ax.set_xticks(range(1, len(evr)+1))
ax.set_ylim(0, max(evr)*1.15)

# Cumulative variance plot
ax2 = axes[1]
ax2.plot(np.arange(1, len(cev)+1), cev, marker="o", lw=2, color="#F58518")
ax2.set_xlabel("Number of PCs")
ax2.set_ylabel("Cumulative variance explained")
ax2.set_title("Cumulative Explained Variance")
ax2.set_xticks(range(1, len(cev)+1))
ax2.set_ylim(0, 1.05)
ax2.grid(alpha=0.25)

# Reference lines at 2, 3, 4 PCs to discuss tradeoffs
for k, col in zip([2,3,4], ["#AA3377", "#44AA99", "#DDCC77"]):
    axes[0].axvline(k, color=col, linestyle="--", alpha=0.7)
    ax2.axvline(k, color=col, linestyle="--", alpha=0.7)
    ax2.text(k+0.05, 0.02, f"k={k}", color=col)

# Annotate % at 2,3,4 PCs on cumulative plot
for k in [2,3,4]:
    ax2.annotate(f"{cev[k-1]*100:.1f}%",
                 xy=(k, cev[k-1]), xytext=(k+0.15, min(0.98, cev[k-1]+0.06)),
                 arrowprops=dict(arrowstyle="-", color="gray", lw=1),
                 fontsize=10, color="black")

plt.tight_layout()
plt.show()

# Optional: save for slides/Canvas
# fig.savefig("pca_scree_cumulative.png", dpi=200, bbox_inches="tight")


# **High-dimensionality data example (n=200, p=100):**
### - Visualizations: Scree, Cumulative Var Explained, and Cross Validated (CV) R² vs PC numbers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# ---- simulate high-dimensional, collinear data ----
rng = np.random.default_rng(42)
n, p = 200, 100       # 200 samples, 100 predictors

# latent biological factors (true dimensionality = 3)
latent1 = rng.normal(0, 1, n)
latent2 = rng.normal(0, 1, n)
latent3 = rng.normal(0, 1, n)

# each feature is a noisy mix of the 3 latents
X = np.zeros((n, p))
for j in range(p):
    w = rng.normal(0, 1, 3)
    X[:, j] = w[0]*latent1 + w[1]*latent2 + w[2]*latent3 + rng.normal(0, 0.3, n)

# true response depends only on latent1 and latent2
y = 2.0*latent1 - 1.5*latent2 + rng.normal(0, 1.0, n)

# ---- standardize features ----
Xz = StandardScaler().fit_transform(X)

# ---- PCA ----
pca = PCA().fit(Xz)
evr = pca.explained_variance_ratio_
cev = np.cumsum(evr)

# ---- OLS (all 100 features) ----
lm = LinearRegression()
r2_full = cross_val_score(lm, Xz, y, cv=5, scoring="r2").mean()

# ---- PCA regression (vary k PCs) ----
max_k = 50
ks = np.arange(1, max_k+1)
r2_pcr = []
for k in ks:
    Xk = pca.transform(Xz)[:, :k]
    r2 = cross_val_score(lm, Xk, y, cv=5, scoring="r2").mean()
    r2_pcr.append(r2)
r2_pcr = np.array(r2_pcr)
k_best = ks[np.argmax(r2_pcr)]
best_r2 = r2_pcr.max()

print(f"Full OLS (100 features) CV R² = {r2_full:.3f}")
print(f"Best PCA regression CV R² = {best_r2:.3f} at k={k_best}")

# ---- stacked plots ----
fig, axes = plt.subplots(3, 1, figsize=(8, 12))

# 1️⃣ Scree plot
ax = axes[0]
ax.bar(np.arange(1, len(evr)+1), evr, color="#4C78A8", alpha=0.9)
ax.set_xlim(0.5, 15.5)
ax.set_xlabel("Principal Component")
ax.set_ylabel("Proportion of variance explained")
ax.set_title("Scree Plot (first ~15 PCs)")
for k, col in zip([3,5,10], ["#AA3377", "#44AA99", "#DDCC77"]):
    ax.axvline(k, color=col, linestyle="--", alpha=0.8)

# 2️⃣ Cumulative variance
ax = axes[1]
ax.plot(np.arange(1, len(cev)+1), cev, lw=2, color="#F58518")
ax.set_xlim(1, 50)
ax.set_ylim(0, 1.03)
ax.set_xlabel("Number of PCs")
ax.set_ylabel("Cumulative variance explained")
ax.set_title("Cumulative Explained Variance")
ax.grid(alpha=0.3)
for k, col in zip([3,5,10], ["#AA3377", "#44AA99", "#DDCC77"]):
    ax.axvline(k, color=col, ls="--", alpha=0.8)
    ax.text(k+0.5, min(0.98, cev[k-1]+0.05), f"{cev[k-1]*100:.1f}%", color=col)

# 3️⃣ Cross-validated R² vs number of PCs
ax = axes[2]
ax.plot(ks, r2_pcr, lw=2, color="#2CA02C", label="PCA regression (k PCs)")
ax.axhline(r2_full, color="crimson", ls="--", lw=2, label="Full OLS (100 features)")
ax.plot([k_best], [best_r2], "o", color="black", label=f"Best k={k_best}")
ax.set_xlabel("Number of PCs (k)")
ax.set_ylabel("5-fold CV R²")
ax.set_title("Generalization Performance: PCR vs Full OLS")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


# **Visualization for high dimensionality data**
## UMAP and t-SNE plots (non-linear dimensionality reduction)
* Both UMAP and t-SNE take your p=100 dimension standardized features (Xz) and find a nonlinear 2D embedding preserving local structure.
* Points with similar hidden factor values (combinations of latent1, latent2, latent3) cluster together.
* Coloring by y (the true response) reveals how that underlying response varies smoothly across the latent manifold.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
try:
    import umap
except ImportError:
    !pip install -q umap-learn
    import umap

# --- regenerate high-dimensional data if needed ---
try:
    Xz, y
except NameError:
    rng = np.random.default_rng(42)
    n, p = 200, 100
    latent1 = rng.normal(0, 1, n)
    latent2 = rng.normal(0, 1, n)
    latent3 = rng.normal(0, 1, n)
    X = np.zeros((n, p))
    for j in range(p):
        w = rng.normal(0, 1, 3)
        X[:, j] = w[0]*latent1 + w[1]*latent2 + w[2]*latent3 + rng.normal(0, 0.3, n)
    y = 2.0*latent1 - 1.5*latent2 + rng.normal(0, 1.0, n)
    Xz = StandardScaler().fit_transform(X)

# --- Dimensionality reduction ---
reducer_umap = umap.UMAP(n_neighbors=20, min_dist=0.1, n_components=2)
#reducer_umap = umap.UMAP(n_neighbors=20, min_dist=0.1, n_components=2, random_state=42)
X_umap = reducer_umap.fit_transform(Xz)

reducer_tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
X_tsne = reducer_tsne.fit_transform(Xz)

# --- Combine both in one figure ---
fig, axes = plt.subplots(1, 2, figsize=(12,5))

# UMAP
sc = axes[0].scatter(X_umap[:,0], X_umap[:,1], c=y, cmap="viridis", s=25, alpha=0.8)
axes[0].set_title("UMAP projection (color = y)")
axes[0].set_xlabel("UMAP-1")
axes[0].set_ylabel("UMAP-2")
axes[0].grid(alpha=0.3)
plt.colorbar(sc, ax=axes[0], label="y (response)")

# t-SNE
sc2 = axes[1].scatter(X_tsne[:,0], X_tsne[:,1], c=y, cmap="viridis", s=25, alpha=0.8)
axes[1].set_title("t-SNE projection (color = y)")
axes[1].set_xlabel("t-SNE-1")
axes[1].set_ylabel("t-SNE-2")
axes[1].grid(alpha=0.3)
plt.colorbar(sc2, ax=axes[1], label="y (response)")

plt.suptitle("UMAP and t-SNE on high-dimensional latent-factor data", fontsize=14)
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()
